Implement a Gradient Boosting model for classification by fine-tuning learning rates, tree depths, and subsampling techniques while using early stopping to prevent overfitting and optimize training efficiency.

In [8]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib
import os
import xgboost as xgb


In [2]:
save_folder = "task31_saved_files"
preprocessor = joblib.load(os.path.join(save_folder, "preprocessor.pkl"))
feature_config = joblib.load(os.path.join(save_folder, "feature_config.pkl"))
x_test = joblib.load(os.path.join(save_folder, "x_test.pkl"))
y_test = joblib.load(os.path.join(save_folder, "y_test.pkl"))
x_train = joblib.load(os.path.join(save_folder, "x_train.pkl"))
y_train = joblib.load(os.path.join(save_folder, "y_train.pkl"))
print("All saved files loaded successfully!")

All saved files loaded successfully!


In [3]:
save_folder = "task31_saved_files"

x_train_processed = preprocessor.transform(x_train)
x_test_processed = preprocessor.transform(x_test)

print("Processed training shape:", x_train_processed.shape)
print("Processed testing shape:", x_test_processed.shape)

Processed training shape: (18786, 3049)
Processed testing shape: (4697, 3049)


In [5]:
print("Training features:", x_train.shape)
print("Training target:", y_train.shape)

print("\nClass distribution:")
print(y_train.value_counts())

print("\nPreprocessor:", type(preprocessor).__name__)

Training features: (18786, 15)
Training target: (18786,)

Class distribution:
recommend_flag
1    15448
0     3338
Name: count, dtype: int64

Preprocessor: ColumnTransformer


In [7]:
print(type(x_train_processed))
print("Is sparse:", hasattr(x_train_processed, "toarray"))

<class 'scipy.sparse._csr.csr_matrix'>
Is sparse: True


In [9]:
print("XGBoost version:", xgb.__version__)

XGBoost version: 3.3.0


Build the baseline model

In [14]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

In [12]:
scale_pos_weight = 15448/3338
print("Scale Pos Weight:", round(scale_pos_weight, 4))

Scale Pos Weight: 4.6279


In [17]:
baseline_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(x_train_processed, y_train)

y_pred_gb = baseline_model.predict(x_test_processed)
y_prob_gb = baseline_model.predict_proba(x_test_processed)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred_gb), 4))
print("Precision:", round(precision_score(y_test, y_pred_gb), 4))
print("Recall:", round(recall_score(y_test, y_pred_gb), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_gb), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_gb), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))

Accuracy: 0.9442
Precision: 0.9754
Recall: 0.9563
F1 Score: 0.9658
ROC-AUC: 0.9795

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       834
           1       0.98      0.96      0.97      3863

    accuracy                           0.94      4697
   macro avg       0.89      0.92      0.91      4697
weighted avg       0.95      0.94      0.95      4697


Confusion Matrix:
[[ 741   93]
 [ 169 3694]]


In [16]:
print(baseline_model.get_params())

{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 100, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 1.0, 'tree_method': None, 'validate_parameters': None, 'verbosity': None, 'obj': 'binary:logistic'}


In [18]:
from sklearn.model_selection import train_test_split

In [20]:
x_train_gb, x_test_gb, y_train_gb, y_test_gb = train_test_split(x_train_processed, y_train, test_size=0.2, random_state=42, stratify=y_train)

print("Training shape:", x_train_gb.shape)
print("Validation shape:", x_test_gb.shape)

print("\nTraining distribution:")
print(y_train_gb.value_counts())

print("\nValidation distribution:")
print(y_test_gb.value_counts())

Training shape: (15028, 3049)
Validation shape: (3758, 3049)

Training distribution:
recommend_flag
1    12358
0     2670
Name: count, dtype: int64

Validation distribution:
recommend_flag
1    3090
0     668
Name: count, dtype: int64


Tune Learning Rate

In [24]:
learning_rates = [0.01, 0.05, 0.1, 0.2]

lr_results = []

for lr in learning_rates:
    model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        subsample=1.0,
        colsample_bytree=1.0,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(x_train_gb, y_train_gb)
    
    y_pred = model.predict(x_test_gb)
    y_prob = model.predict_proba(x_test_gb)[:, 1]
    accuracy = accuracy_score(y_test_gb, y_pred)
    
    lr_results.append({
        'Learning Rate': lr,
        'Accuracy': accuracy_score(y_test_gb, y_pred),
        'F1 Score': f1_score(y_test_gb, y_pred),
        'ROC-AUC': roc_auc_score(y_test_gb, y_prob)
    })

lr_results_df = pd.DataFrame(lr_results)

print(lr_results_df.round(4))

   Learning Rate  Accuracy  F1 Score  ROC-AUC
0           0.01    0.9141    0.9501   0.9772
1           0.05    0.9364    0.9607   0.9786
2           0.10    0.9385    0.9622   0.9794
3           0.20    0.9383    0.9621   0.9795


Hyperparameter Tuning

In [32]:
# Base XGBoost model
from sklearn.model_selection import RandomizedSearchCV


gb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=1
)

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 5, 7],
    'subsample': [0.6, 0.8, 1.0],
    'n_estimators': [100, 200, 300, 500],
    'min_child_weight': [1, 3, 5]
}

random_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(x_train_gb, y_train_gb)


print("Best Parameters:")
# print(random_search.best_params_)
print(random_search.best_params_)
print("\nBest CV F1 Score:")
print(round(random_search.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Parameters:
{'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.05}

Best CV F1 Score:
0.9663


In [35]:
best_params = random_search.best_params_.copy()
best_params.pop('n_estimators')

final_gb_model = xgb.XGBClassifier(
    **best_params,
    n_estimators=1000,
    objective='binary:logistic',
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

final_gb_model.fit(x_train_gb, y_train_gb, eval_set=[(x_test_gb, y_test_gb)], verbose=False)

print("Best Iteration:", final_gb_model.best_iteration)
print("Best Validation Score:", final_gb_model.best_score)


Best Iteration: 133
Best Validation Score: 0.13340423270224389


In [36]:
# Predictions
y_pred_final = final_gb_model.predict(x_test_processed)
y_prob_final = final_gb_model.predict_proba(x_test_processed)[:, 1]

# Evaluation
print("Final Gradient Boosting Results")
print("-------------------------------")

print("Accuracy:", round(accuracy_score(y_test, y_pred_final), 4))
print("Precision:", round(precision_score(y_test, y_pred_final), 4))
print("Recall:", round(recall_score(y_test, y_pred_final), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_final), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_final), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

Final Gradient Boosting Results
-------------------------------
Accuracy: 0.9417
Precision: 0.9716
Recall: 0.957
F1 Score: 0.9643
ROC-AUC: 0.9792

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       834
           1       0.97      0.96      0.96      3863

    accuracy                           0.94      4697
   macro avg       0.89      0.91      0.90      4697
weighted avg       0.94      0.94      0.94      4697


Confusion Matrix:
[[ 726  108]
 [ 166 3697]]


### Conclusion

Successfully implemented an XGBoost-based Gradient Boosting classification model to predict customer recommendation behaviour. Hyperparameter optimization was performed using RandomizedSearchCV by experimenting with different learning rates, maximum tree depths, and subsampling techniques. The best configuration was obtained with a learning rate of 0.05, maximum depth of 7, and subsample ratio of 0.8. Early stopping was also implemented, identifying iteration 133 as the optimal training point with a validation log loss of 0.1334.

The final tuned model achieved 94.17% accuracy, 97.16% precision, 95.70% recall, 96.43% F1-score, and 97.92% ROC-AUC on the test dataset. The results indicate strong classification performance and good ability to distinguish between customers who recommend and do not recommend the products. However, the tuned model performed slightly below the baseline model on most test metrics, showing that hyperparameter tuning does not always guarantee improved performance.

Overall, this task demonstrated the practical application of Gradient Boosting, hyperparameter optimization, subsampling, and early stopping for building and evaluating an effective classification model.

In [39]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

# Save baseline XGBoost model
joblib.dump(baseline_model, "saved_models/baseline_xgb_model.pkl")

# Save tuned XGBoost model
joblib.dump(final_gb_model, "saved_models/tuned_xgb_model.pkl")

print("All models and datasets saved successfully!")

All models and datasets saved successfully!
